# Lesson 8.4 — Unit 8 Recap, Module 9 Close, Handoff to Module 10

The whole module in one call: harvest_row runs all eight units — perception to recovery to row orchestration — with graceful degradation.

In [1]:
import numpy as np
from modules.module09.integration import (GreenhouseWorld, Fruit, model_perception,
    understand, to_configuration, recover, harvest_row)
W = GreenhouseWorld.demo_row(n=6, seed=1)
DETS = model_perception(W, rng=np.random.default_rng(7))
TGT = understand(DETS, W)[1]            # nearest ripe reachable target
kick = lambda t, j: 60.0 if (j == 0 and t > 0.3) else 0.0
checks = []
ripe = {f.fid for f in W.fruit if f.ripe}
# clean: the full system completes the row
clean = harvest_row(W)
print('CLEAN        -> harvested=%d/%d skipped=%d complete=%s'
      % (len(clean['harvested']), len(ripe), len(clean['skipped']), clean['complete']))
checks.append(clean['complete'] and set(clean['harvested']) == ripe)
victim = clean['harvested'][2]
vxy = next(d['xy'] for d in DETS if d['id'] == victim)


CLEAN        -> harvested=5/5 skipped=0 complete=True


In [2]:
# transient injection -> graceful recovery
rec = harvest_row(W, inject={victim: dict(disturbance=lambda a: (kick if a == 0 else None))})
vp = next(p for p in rec['picks'] if p['target'] == victim)
print('TRANSIENT    -> victim recovered=%s | harvested=%d complete=%s'
      % (vp.get('recovered'), len(rec['harvested']), rec['complete']))
checks.append(vp.get('recovered') is True and rec['complete'])
# deterministic injection -> graceful skip
det = harvest_row(W, inject={victim: dict(obstacle=(vxy, 0.25))})
print('DETERMINISTIC-> skipped=%s complete=%s' % (det['skipped'], det['complete']))
checks.append(victim in det['skipped'] and det['complete'])
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('MODULE 9 COMPLETE: eight units -> one self-healing harvester. Next: Module 10 Digital Twin.')
print('All checks passed.')


TRANSIENT    -> victim recovered=True | harvested=5 complete=True


DETERMINISTIC-> skipped=['F0'] complete=True
3/3 checks passed.
MODULE 9 COMPLETE: eight units -> one self-healing harvester. Next: Module 10 Digital Twin.
All checks passed.
